In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
data = spark.table("electronics_retailer_clg.bronze.customers")

In [0]:
import pyspark.sql.functions as F

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("birthday", data)
display(data.select(F.count("*")))

In [0]:
def state_null_handle(df):

    mapping_df = df.filter(F.col("state").isNotNull()) \
               .select("state_code", "state") \
               .dropDuplicates(["state_code"])
    return df.alias("a").join(
        mapping_df.alias("b"), 
        on="state_code", 
        how="left"
    ).select(
        F.col("a.customerkey"),
        F.col("a.gender"),
        F.col("a.name"),
        F.col("a.city"),
        F.col("a.state_code"),
        F.coalesce(F.col("a.state"), F.col("b.state")).alias("state"),
        F.col("a.zip_code"),
        F.col("a.country"),
        F.col("a.continent"),
        F.col("a.birthday")
    )

data = state_null_handle(data)

In [0]:

def zip_code_null_handle(df):
    mapping_df = df.filter(F.col("zip_code").isNotNull())\
                   .select("city", "state", "zip_code")\
                   .dropDuplicates(["city", "state"])

    return df.alias("a").join(
        mapping_df.alias("b"), 
        on=["city", "state"], 
        how="left"
    ).select(
        "a.customerkey",
        "a.gender",
        "a.name",
        "a.city",
        "a.state_code",
        "a.state",
        F.coalesce(F.col("a.zip_code"), F.col("b.zip_code")).alias("zip_code"),
        "a.country",
        "a.continent",
        "a.birthday"
    )
data = zip_code_null_handle(data)

In [0]:

def dataTypeHandle(df,colname,dataType):
    if dict(df.dtypes)[colname] == 'string':
        return df.withColumn(colname,F.trim(F.col(colname)).try_cast(dataType))
    if dict(df.dtypes)[colname] == dataType:
        return df.withColumn(colname,F.col(colname))


data = dataTypeHandle(data,'customerkey','int')
data = dataTypeHandle(data,'gender','string')
data = dataTypeHandle(data,'name','string')
data = dataTypeHandle(data,'city','string')
data = dataTypeHandle(data,'state_code','string')
data = dataTypeHandle(data,'state','string')
data = dataTypeHandle(data,'zip_code','string')
data = dataTypeHandle(data,'country','string')
data = dataTypeHandle(data,'continent','string')
data = dataTypeHandle(data,'birthday','date')
data = data.filter(F.col("customerkey").isNotNull())

In [0]:
display(data)
data.printSchema()


# WRITE TO SILVER
data.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("electronics_retailer_clg.silver.customers")

In [0]:
# from pyspark.sql.functions import col, trim, when

# # READ BRONZE TABLE
# df = spark.table("electronics_retailer_clg.bronze.customers")

# # CLEAN COLUMN NAMES
# df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

# # TRIM SPACES
# for c in df.columns:
#     df = df.withColumn(c, trim(col(c)))

# # REMOVE INVALID PRIMARY KEYS
# df = df.filter(col("customerkey").isNotNull())

# # FIX DATA TYPES
# df = df.withColumn("customerkey", col("customerkey").cast("int"))

# # HANDLE NULL VALUES
# df = df.fillna({
#     "gender": "unknown",
#     "continent": "unknown"
# })

# # STANDARDIZE GENDER
# df = df.withColumn(
#     "gender",
#     when(col("gender").isin("Male", "Female"), col("gender"))
#     .otherwise("Unknown")
# )

# # REMOVE DUPLICATES
# df = df.dropDuplicates(["customerkey"])


# # KEEP ONLY REQUIRED COLUMNS
# df = df.select(
#     "customerkey",
#     "gender",
#     "continent"
# )


# display(df)
# df.printSchema()


# # WRITE TO SILVER
# df.write.format("delta") \
#     .mode("overwrite") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("electronics_retailer_clg.silver.customers")

# print("Customers cleaned & optimized successfully")